**Imports**

In [17]:
import os
from planet import Session, DataClient, OrdersClient, order_request
from shapely.geometry import shape

**Authentication**

In [ ]:
os.environ['PL_API_KEY'] = ""  #fill in the api key here
client = DataClient(Session())

## 🛰️ PlanetScope Catalogue Query & Coverage Analysis

This script queries the Planet **Data API** to find all available **PSScene** (Daily PlanetScope) products within the specified date range and AOI. 

### Key Features:
* **Quality Filtering:** Only returns scenes with less than **20% cloud cover**.
* **Strip ID Grouping:** Satellite imagery is captured in long strips. By grouping results by `strip_id`, we identify "sister" scenes captured during the same orbit that will stitch together seamlessly.
* **Spatial Intersection Check:** Uses `shapely` to calculate the exact percentage of your AOI covered by each scene. This prevents downloading "empty" space or partial captures.

### How to Interpret the Output:
1. **Find 100% Coverage:** Look for a single ID with **100% AOI Coverage** for the simplest download.
2. **Handle Partial Coverage:** If no single scene hits 100%, look for multiple IDs within the **same Strip ID** that sum to 100%. These should be ordered together.
3. **Check Cloud %:** Note that cloud cover is reported for the *entire* scene; a small cloud % might not actually affect your specific AOI.

---
**Next Step:** Copy the desired **ID(s)** from the list below to use in the **Orders API** cell.

In [18]:
# 1. Define AOI
aoi = {
    "type": "Polygon",
    "coordinates": [[
        [22.058141667179701, 37.108059967364362], 
        [22.058141667179701, 37.008059967364360], 
        [22.158141667179702, 37.008059967364360], 
        [22.158141667179702, 37.108059967364362], 
        [22.058141667179701, 37.108059967364362]  
    ]]
}

# 2. Search Filter
search_filter = {
    "type": "AndFilter",
    "config": [
        {"type": "GeometryFilter", "field_name": "geometry", "config": aoi},
        {"type": "DateRangeFilter", "field_name": "acquired", "config": {"gte": "2026-01-01T00:00:00Z", "lte": "2026-01-30T00:00:00Z"}},
        {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lte": 0.2}} 
    ]
}

async with Session() as sess:
    data_client = DataClient(sess)
    aoi_geom = shape(aoi)
    aoi_area = aoi_geom.area
    
    strips = {}
    
    async for item in data_client.search(name="detailed_search", search_filter=search_filter, item_types=["PSScene"]):
        props = item['properties']
        s_id = props.get('strip_id')
        
        # Calculate coverage of your AOI
        scene_geom = shape(item['geometry'])
        intersection = aoi_geom.intersection(scene_geom)
        coverage_pct = (intersection.area / aoi_area) * 100
        
        if s_id not in strips:
            strips[s_id] = []
        
        strips[s_id].append({
            'id': item['id'],
            'clouds': props.get('cloud_cover', 0) * 100,
            'coverage': coverage_pct,
            'acquired': props.get('acquired')
        })

    # Print the results in a readable format
    for s_id, items in strips.items():
        print(f"--- Strip ID: {s_id} ---")
        for i in items:
            print(f"  ID: {i['id']}")
            print(f"     ☁️ Clouds: {i['clouds']:.1f}% | 🗺️ AOI Coverage: {i['coverage']:.1f}%")
        print(f"  Total items: {len(items)}\n")
        
# After running this, look at the list and pick the IDs you want.

--- Strip ID: 8560384 ---
  ID: 20260127_095156_62_24de
     ☁️ Clouds: 5.0% | 🗺️ AOI Coverage: 37.8%
  ID: 20260127_095158_69_24de
     ☁️ Clouds: 3.0% | 🗺️ AOI Coverage: 44.4%
  Total items: 2

--- Strip ID: 8560451 ---
  ID: 20260127_095301_56_251b
     ☁️ Clouds: 2.0% | 🗺️ AOI Coverage: 99.7%
  ID: 20260127_095259_47_251b
     ☁️ Clouds: 6.0% | 🗺️ AOI Coverage: 26.2%
  Total items: 2

--- Strip ID: 8560375 ---
  ID: 20260127_094658_21_2549
     ☁️ Clouds: 5.0% | 🗺️ AOI Coverage: 23.2%
  ID: 20260127_094700_48_2549
     ☁️ Clouds: 11.0% | 🗺️ AOI Coverage: 4.2%
  Total items: 2

--- Strip ID: 8560213 ---
  ID: 20260127_092016_96_255d
     ☁️ Clouds: 8.0% | 🗺️ AOI Coverage: 21.3%
  ID: 20260127_092014_59_255d
     ☁️ Clouds: 4.0% | 🗺️ AOI Coverage: 100.0%
  Total items: 2

--- Strip ID: 8553866 ---
  ID: 20260124_095158_21_250b
     ☁️ Clouds: 14.0% | 🗺️ AOI Coverage: 84.7%
  ID: 20260124_095156_09_250b
     ☁️ Clouds: 11.0% | 🗺️ AOI Coverage: 47.3%
  Total items: 2

--- Strip ID: 854

## 📦 Step 2: Submit Clipped Composite Order

Once you have identified the best scene(s) from the catalogue search, this step submits a processing request to the **Orders API**. 

### What happens in this step:
* **Product Selection:** Requests the `analytic_sr_udm2` bundle (Surface Reflectance data + Usable Data Mask).
* **Clipping:** Planet's server "cookie-cuts" the imagery to the exact boundary of your AOI polygon.
* **Compositing:** If multiple IDs are provided (e.g., sister scenes from the same strip), they are merged into a single, seamless GeoTIFF file.
* **Delivery:** An order is created on Planet's backend. Processing usually takes between 1 to 5 minutes depending on the server load.

---
**Note:** After running this cell, copy the **Order ID** printed below to use in the final **Download** step.

In [21]:
# --- USER INPUT ---
# Paste the IDs you want to download here (one or many)
selected_ids = ["20260113_092244_55_257a", "20260113_092242_16_257a"] 
# ------------------

async with Session() as sess:
    orders_client = OrdersClient(sess)

    request = order_request.build_request(
        name="Manual_Selection_Composite",
        products=[
            order_request.product(
                item_ids=selected_ids,
                product_bundle="analytic_8b_sr_udm2",
                item_type="PSScene"
            )
        ],
        tools=[
            order_request.clip_tool(aoi),    # This ensures it fits your tile perfectly
            order_request.composite_tool()  # This merges multiple IDs into one file
        ]
    )

    order = await orders_client.create_order(request)
    print(f"Order created! ID: {order['id']}")

Order created! ID: 12bf9c87-7672-4a6f-91f0-d6e303cc9453


## 📥 Step 3: Monitor & Download Order

Planet processes orders asynchronously. This cell monitors the order status and automatically downloads the files once processing is complete.

**Note:** You can run this cell immediately after submitting your order. The script will "poll" the Planet API every 10 seconds until the files are ready. Processing usually takes several minutes. 

### What this script does:
* **Polls for Status:** Uses `.wait()` to check the order status every few seconds.
* **Detects Completion:** Once the state reaches `success`, it triggers the download.
* **Local Storage:** Saves the resulting `.tif` (imagery) and `.json` (metadata) files into a local folder named `planet_downloads`.

In [22]:
# 1. Define where you want to save the files
download_dir = 'planet_downloads'
os.makedirs(download_dir, exist_ok=True)

async with Session() as sess:
    cl = OrdersClient(sess)
    
    # 2. Wait for the order to be processed
    # This will print updates until it's finished (success) or failed
    print(f"Waiting for order {order['id']} to complete...")
    state = await cl.wait(order_id=order['id'], delay=10, max_attempts=100)
    
    if state == 'success':
        # 3. Download the files to the specified directory
        print("Order is ready! Starting download...")
        filenames = await cl.download_order(
            order_id=order['id'], 
            directory=download_dir, 
            overwrite=True
        )
        
        print("\n✅ Download Complete!")
        for f in filenames:
            print(f"Saved: {f}")
    else:
        print(f"❌ Order failed with state: {state}")

Waiting for order 12bf9c87-7672-4a6f-91f0-d6e303cc9453 to complete...
Order is ready! Starting download...

✅ Download Complete!
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/20260113_092242_16_257a_metadata.json
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/20260113_092242_16_257a_3B_AnalyticMS_8b_metadata_clip.xml
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/composite_udm2.tif
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/20260113_092244_55_257a_metadata.json
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/composite_metadata.json
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/composite.tif
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/20260113_092244_55_257a_3B_AnalyticMS_8b_metadata_clip.xml
Saved: planet_downloads/12bf9c87-7672-4a6f-91f0-d6e303cc9453/manifest.json
